# 03 · Limpieza Dura: Variables Inútiles y Diagnóstico por Segmento
## Módulo 3 — Feature Engineering & Modelado Supervisado

---

### 🎯 Objetivo del Cuaderno
Ejecutar la poda dura (*Hard Pruning*) sobre el conjunto de características de entrenamiento ($X_{train}$), descartando variables que no aportan señal predictiva útil por:
1. **Varianza Cero (Cardinalidad 1):** Variables constantes en toda la muestra.
2. **Varianza Casi Nula (*Near-Zero Variance*):** Una sola categoría domina más del 95% de los registros.
3. **Exceso de Nulos:** Atributos con más del 95% de datos ausentes.

### ⚠️ Diagnóstico por Segmento (La Vuelta de Tuerca)
Inspirados en la lección de arquitectura, evaluaremos las variables no solo de forma global, sino **dentro de cada Clúster (`Cluster 0` y `Cluster 1`)**:
* Si una variable es constante en un clúster pero varía en el otro, **esa constancia es la regla estructural del segmento** (no es ruido, es señal) y se **preserva**.
* Solo se descartan variables si son inútiles globalmente o si están verdaderamente muertas sin aportar diferenciación.

🔒 **Regla de No-Contaminación:** La lista de columnas a descartar se aprende e identifica **exclusivamente sobre $X_{train}$**. Luego, esa misma lista se elimina de $X_{test}$ sin volver a evaluar métricas sobre el conjunto de prueba.

In [1]:
# ==============================================================================
# BLOQUE 1: CARGA DE ARTEFACTOS (TRAIN Y TEST)
# ==============================================================================
import pandas as pd
import numpy as np

# Configuración visual
pd.set_option('display.max_columns', None)

# Carga de archivos oficiales
ruta_train = 'data_processed/train_ecommerce.csv'
ruta_test = 'data_processed/test_ecommerce.csv'

train_df = pd.read_csv(ruta_train)
test_df = pd.read_csv(ruta_test)

target_col = 'Cluster'

# Separamos X e y para Train y Test
X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print(f"📦 Train cargado: {X_train.shape[0]} filas × {X_train.shape[1]} columnas.")
print(f"🔒 Test cargado (aislado): {X_test.shape[0]} filas × {X_test.shape[1]} columnas.")

📦 Train cargado: 8086 filas × 61 columnas.
🔒 Test cargado (aislado): 2022 filas × 61 columnas.


## 2. Diagnóstico de Varianza Global y por Segmento (Clúster)

Auditamos la frecuencia de la categoría dominante para cada variable:
* **Nivel Global ($X_{train}$):** Dominancia $\ge 95\%$.
* **Nivel Segmento ($X_{train} \mid \text{Cluster} = k$):** Analizamos si la homogeneidad obedece a la estructura del clúster o a falta de información.

In [2]:
# ==============================================================================
# BLOQUE 2 Y 3: DIAGNÓSTICO DE VARIANZA GLOBAL Y POR SEGMENTO
# ==============================================================================

def diagnosticar_varianza_por_segmento(X, y, umbral_dominancia=0.95):
    resultados_global = []
    resultados_segmento = []
    
    # 1. Diagnóstico Global
    for col in X.columns:
        vc = X[col].value_counts(dropna=False, normalize=True)
        top_pct = vc.iloc[0] if len(vc) > 0 else 1.0
        top_cat = vc.index[0] if len(vc) > 0 else np.nan
        
        es_constante = (X[col].nunique(dropna=False) <= 1)
        es_quasi_constante = (top_pct >= umbral_dominancia)
        
        resultados_global.append({
            'variable': col,
            'unicos_global': X[col].nunique(dropna=False),
            'top_cat_global': top_cat,
            'dominancia_global': round(top_pct * 100, 2),
            'descarte_global': es_constante or es_quasi_constante
        })
        
        # 2. Diagnóstico por Segmento
        for cluster_id in sorted(y.unique()):
            mask = (y == cluster_id)
            X_seg = X[mask]
            
            vc_seg = X_seg[col].value_counts(dropna=False, normalize=True)
            top_pct_seg = vc_seg.iloc[0] if len(vc_seg) > 0 else 1.0
            top_cat_seg = vc_seg.index[0] if len(vc_seg) > 0 else np.nan
            
            resultados_segmento.append({
                'variable': col,
                'segmento': f'Cluster {cluster_id}',
                'top_cat_segmento': top_cat_seg,
                'dominancia_segmento': round(top_pct_seg * 100, 2),
                'es_homogenea_en_seg': (top_pct_seg >= umbral_dominancia)
            })
            
    df_global = pd.DataFrame(resultados_global)
    df_segmento = pd.DataFrame(resultados_segmento)
    
    return df_global, df_segmento

df_diag_global, df_diag_seg = diagnosticar_varianza_por_segmento(X_train, y_train, umbral_dominancia=0.95)

# Filtramos variables candidatas a descarte global
candidatas_descarte = df_diag_global[df_diag_global['descarte_global']]['variable'].tolist()

print("🔍 RESULTADOS DEL DIAGNÓSTICO DE VARIANZA:")
print(f"· Variables constantes o quasi-constantes a nivel GLOBAL (>95% dominancia): {len(candidatas_descarte)}")

# Verificamos si hay variables altamente homogéneas en algún segmento
homogeneas_seg = df_diag_seg[df_diag_seg['es_homogenea_en_seg']]
print(f"· Casos de variables con >95% dominancia dentro de un segmento específico: {len(homogeneas_seg)}")

if len(candidatas_descarte) > 0:
    print("\n⚠️ Variables identificadas para Poda Dura:")
    display(df_diag_global[df_diag_global['descarte_global']])
else:
    print("\n✅ Ninguna variable presenta varianza cero o near-zero global. Todas las variables superan el filtro de diversidad.")

if len(homogeneas_seg) > 0:
    print("\n📌 Muestra de variables con homogeneidad en algún segmento (Estructura de Clúster):")
    display(homogeneas_seg.head(10))

🔍 RESULTADOS DEL DIAGNÓSTICO DE VARIANZA:
· Variables constantes o quasi-constantes a nivel GLOBAL (>95% dominancia): 0
· Casos de variables con >95% dominancia dentro de un segmento específico: 10

✅ Ninguna variable presenta varianza cero o near-zero global. Todas las variables superan el filtro de diversidad.

📌 Muestra de variables con homogeneidad en algún segmento (Estructura de Clúster):


,variable,segmento,top_cat_segmento,dominancia_segmento,es_homogenea_en_seg
33,how_you_heard_about_survey_banner,Cluster 1,No,95.13,True
34,how_you_heard_about_survey_friend,Cluster 0,No,98.19,True
36,how_you_heard_about_survey_mailing_list,Cluster 0,No,98.38,True
46,major_geographical_location,Cluster 0,USA,97.16,True
58,primary_language,Cluster 0,English,98.30,True
80,not_purchasing_not_option,Cluster 0,No,96.85,True
92,not_purchasing_too_complicated,Cluster 0,No,96.01,True
106,who_pays_for_access_parents,Cluster 0,No,98.36,True
108,who_pays_for_access_school,Cluster 0,No,99.00,True
116,clean_country,Cluster 0,United_States,97.16,True


## 4. Aplicación de Poda Dura y Exportación de Datasets Limpios

Siguiendo el principio estricto de **No Contaminación (*Train-Only*)**:
1. Se consolida la lista oficial de variables a descartar a partir del diagnóstico de $X_{train}$. Dado que $0$ variables superaron el umbral global de dominancia del $95\%$, la lista de poda está vacía (`cols_a_eliminar = []`).
2. Se aplica la transformación idéntica tanto a `train` como a `test`.
3. Se registran los reportes en formato ejecutable y se exportan las bases procesadas (`03_train_ecommerce.csv` y `03_test_ecommerce.csv`).

In [3]:
# ==============================================================================
# BLOQUE 4: PODA OFICIAL Y EXPORTACIÓN DE DATASETS LIMPIOS
# ==============================================================================
import json
import os

# 1. Lista oficial de columnas a eliminar (aprendida exclusivamente sobre Train)
cols_a_eliminar = candidatas_descarte  # Lista vacía []

print(f"✂️ Columnas a eliminar por Poda Dura Global: {len(cols_a_eliminar)}")

# 2. Aplicamos la eliminación (si existiera alguna) en Train y Test
X_train_clean = X_train.drop(columns=cols_a_eliminar)
X_test_clean = X_test.drop(columns=cols_a_eliminar)

# Reintegramos el Target para exportar los datasets completos
train_03 = X_train_clean.copy()
train_03[target_col] = y_train

test_03 = X_test_clean.copy()
test_03[target_col] = y_test

# 3. Guardado de bases procesadas
os.makedirs('data_processed', exist_ok=True)
os.makedirs('reports', exist_ok=True)

train_03.to_csv('data_processed/03_train_ecommerce.csv', index=False)
test_03.to_csv('data_processed/03_test_ecommerce.csv', index=False)

# 4. Guardado del reporte de decisiones JSON (Arquitectura del Profesor)
reporte_poda = {
    'total_variables_iniciales': X_train.shape[1],
    'columnas_eliminadas_varianza_nula': cols_a_eliminar,
    'total_variables_finales': X_train_clean.shape[1],
    'variables_homogeneas_por_segmento': list(df_diag_seg[df_diag_seg['es_homogenea_en_seg']]['variable'].unique())
}

with open('reports/columnas_descartadas.json', 'w') as f:
    json.dump(reporte_poda, f, indent=4)

print("="*75)
print("✅ DATASETS DE LIMPIEZA DURA EXPORTADOS CON ÉXITO:")
print(f"· Train Módulo 03: {train_03.shape[0]} filas × {train_03.shape[1]} columnas")
print(f"· Test Módulo 03:  {test_03.shape[0]} filas × {test_03.shape[1]} columnas")
print("· Reporte guardado en: 'reports/columnas_descartadas.json'")
print("="*75)

✂️ Columnas a eliminar por Poda Dura Global: 0
✅ DATASETS DE LIMPIEZA DURA EXPORTADOS CON ÉXITO:
· Train Módulo 03: 8086 filas × 62 columnas
· Test Módulo 03:  2022 filas × 62 columnas
· Reporte guardado en: 'reports/columnas_descartadas.json'


## 5. Conclusiones del Diagnóstico y Poda Dura

El análisis de varianza y homogeneidad por segmento sobre $X_{train}$ aporta las siguientes conclusiones para la arquitectura del pipeline:

---

### A. Diagnóstico de Varianza Global (Poda Dura)
* **Preservación Total de Características:** Ninguna variable presentó varianza cero o dominancia mayor al $95\%$ a nivel global. El $100\%$ de las variables explicativas originales posee diversidad suficiente para alimentar los modelos predictivos.
* **Integridad del Conjunto de Características:** El espacio predictivo conserva sus $X$ variables intactas, listas para la fase de detección de observaciones atípicas.

---

### B. Estructura de Segmento vs. Ruido (La Regla del Rescate)
* **Identificación de Homogeneidad en Segmentos:** Se detectaron $10$ variables con dominancia $>95\%$ concentrada en el **Clúster 0 (Consumidor Cauto)**. Destacan `clean_country` ($97.16\%$ Estados Unidos) y `primary_language` ($98.30\%$ Inglés).
* **Fundamento de Negocio:** Eliminar estas variables por su alta homogeneidad en el Clúster 0 habría sido un error grave de modelado. La dominancia del idioma inglés y la procedencia estadounidense en el Clúster 0 no es "varianza muerta", sino **la identidad demográfica estructural que define al segmento**. El filtro global las rescata correctamente como constantes significativas.

---

### C. Garantía Metodológica de No Contaminación (*Train-Only*)
* La evaluación de varianza y la regla de transformación se calcularon **estrictamente sobre $X_{train}$**.
* El conjunto de prueba ($X_{test}$) fue transformado mediante el mismo pipeline sin ser consultado en la fase de diagnóstico, garantizando un entorno libre de *Data Leakage*.

**🏁 Siguiente Paso:**
Con la base de características purgada y validada, se procede al cuaderno **`04_outliers_mv.ipynb`**, donde aplicaremos **detección multivariada de anomalías** utilizando el algoritmo **Isolation Forest** sobre el subespacio de coordenadas factoriales del MCA.